# 07 · Глубокий аудит данных, методов, кода и выводов
## tl;dr
Исправлены агрегация hierarchy-ансамбля, баланс независимых патологических записей, вложенный выбор postprocessing и критерий калибровки. Все шесть семейств повторно прошли полную сетку. Основной экспорт: **C_ovr/single**; shortlist **C_ovr/single, D_hierarchy/single**. Test macro-F1 **0.8538**, recall патологии **91.4%**, AP **0.9103**. Прежние результаты сохранены. Источник и train-only scalers прошли независимую проверку. Уникальное универсальное превосходство и клиническая готовность не установлены.

## Context & Methods

Область аудита: исходная XLS-книга и primary data, ноутбуки 01–06, подготовка, nested CV, калибровка/пороги, soft voting, классификационные и кластеризационные метрики, сохранение моделей/кэша, существенные выводы. Проверки выполнены из исходных строк и сохранённых прогнозов, не только повторным вызовом автора метрик. Проверка повторяющихся графиков исторических ноутбуков выборочная; все новые существенные графики проверяются после исполнения. Взаимодействующих dashboard-контролов нет.

### Критические выводы и исправления

| Приоритет | Наблюдение и значение | Исправление / предел |
|---|---|---|
| P1 | Для hierarchy усреднялись a,b до их произведения; это другой ансамбль | В v2 soft voting трёх вероятностей каждого участника, отдельная калибровка среднего; добавлен независимый арифметический тест |
| P1 | При ровных числах патологических сегментов старый outer-фолд имел лишь 2 патологических файла; calibration holdout — 1 | Фолды балансируют и class segments, и class-presence files; таблицы поддержки сохраняются |
| P1 | В v1 режим postprocessing выбирался после просмотра всех внешних результатов и затем менялся на validation | В v2 выбор калибровки/порога входит внутрь каждой внешней обучающей части; внешний holdout оценивает всю процедуру |
| P2 | Калибровку оценивали преимущественно по macro-F1 | Теперь метод калибровки выбирается по cross-fitted log loss, порог решения отдельно по macro-F1 |
| P2 | Число сегментов переоценивает независимую опору, различия A/D по test не доказывают уникального победителя | Сохранена основная segment-метрика; добавлены file-balanced показатели и paired file bootstrap, shortlist выбирается по развитию модели |
| P2 | Старый cache fingerprint не учитывал live guard source и версии | Новая версия включает исходники guard/алгоритма/фолдов, данные/шаблон подготовки и runtime; результат имеет проверяемые SHA-256 |
| P2 | UMAP transform меняет часть назначений при другом batch | Оставлен исследовательским методом; проверенные 1/20 изменений GMM не скрыты, не экспортируется как supervised champion |
| P2 | Высокий ARI/44.4% патологии AE+GMM опирается на 2 патологических файла и неустойчив к seed | Не используется для выбора классификатора; показатели и ограничения 06 сохранены, добавлена общая интерпретация |

### Что подтверждено

- Все 2126×21 значений независимо восстановлены из `Raw Data`: события AC/FM/UC/DL/DS/DP поделены на e−b, остальные признаки сопоставлены напрямую. Расхождений нет.
- Нет пропусков, пересечений исходных файлов и feature duplicates между train/validation/test. Нули имеют содержательный смысл. Скейлеры совпадают с новым fit только на train; общий fit дал бы отличия до .3214 в проверенной матрице.
- Контракт входа проверяет ровно 21 признак и запрещает неподдерживаемые пропуски/нечисловые значения/неверные категории. Импутация по отсутствующему в train сценарию не добавлена.
- Метки NSP/CLASS не входят в признаки; scaler и UMAP/AE не fit на test. Group early stopping AE корректен. Не обнаружена прямая утечка test в обучение исторических деревьев.
- Старые test macro-F1/recall/AP пересчитаны независимо и совпали. Исправление ансамбля не меняет старую одиночную модель; старая оценка ансамбля признана исторической.

Старая версия 02/03/05 и плана — `history/v1/`. 04 и 06 сохранены побайтно, включая отрицательные результаты; их методы не стали клиническими моделями. Текущие рабочие точки и экспорт — только audited v2.

In [3]:
from pathlib import Path
import sys,json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.training.ctg_audit import (AUDIT,EXPORT,context,CTG_FEATURES,score,soft_vote)
data,fingerprint,cache=context()
sns.set_theme(style='whitegrid',font_scale=1.05)
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.figsize':(10,5)})
def show_save(name):
    plt.gcf().savefig(AUDIT/name,dpi=140,bbox_inches='tight')
    plt.show()

## Data · доказательства аудита

In [5]:
for name in ['ctg_supervised_audit.json','ctg_data_cluster_audit.json']:
    evidence=json.loads((AUDIT/name).read_text())
    print(name)
    if 'findings' in evidence:
        display(pd.DataFrame([{'severity':r['severity'],'issue':r.get('issue',r.get('id'))} for r in evidence['findings']]))
display(pd.read_csv(AUDIT/'outer_fold_support.csv'))
display(pd.read_csv(AUDIT/'validation_fold_support.csv'))

ctg_supervised_audit.json
ctg_data_cluster_audit.json


,severity,issue
0,P1,Averaging hierarchical conditional heads is no...
1,P1,Row-stratified groups yield poor independent-p...
2,P2,Observed A versus D difference does not establ...
3,P2,Current random-segment estimand weights long r...
4,P2,Cache fingerprint omits preprocessing source a...
5,P2,No independent ensemble arithmetic oracle or m...


,severity,issue
0,P2,UMAP_BATCH_SENSITIVITY
1,P2,INDEPENDENT_POSITIVE_SUPPORT
2,P2,CACHE_SOURCE_DEPENDENCY
3,P2,AE_CLUSTER_INSTABILITY
4,P3,NOTEBOOK01_WIDTH_WORDING


,fold,n_fit,n_hold,source_files,normal_segments,normal_files,suspect_segments,suspect_files,path_segments,path_files
0,0,1088,272,45,211,36,39,16,22,6
1,1,1086,274,45,212,35,37,16,25,6
2,2,1089,271,46,212,35,38,16,21,7
3,3,1089,271,46,212,36,38,17,21,7
4,4,1088,272,45,210,35,39,16,23,7


,fold,n_fit,n_hold,source_files,normal_segments,normal_files,suspect_segments,suspect_files,path_segments,path_files
0,0,230,111,19,90,14,15,7,6,2
1,1,230,111,19,89,14,16,6,6,3
2,2,222,119,19,87,15,15,6,17,2


## Results · текущие лучшие процедуры

In [7]:
nested=pd.read_csv(AUDIT/'nested_summary.csv');test=pd.read_csv(AUDIT/'test_metrics.csv')
lock=json.loads((AUDIT/'locked_decision.json').read_text())
display(nested.round(4));display(test[test.shortlisted].round(4))
display(pd.read_csv(AUDIT/'test_paired_intervals.csv').round(4))
print('Рекомендуемый основной bundle:',EXPORT/'champion.joblib')
print('Shortlist:',lock['shortlist'])

Рекомендуемый основной bundle: /Users/nick/Desktop/projects/femfedora/models/exported/ctg_uci/audited_v2/champion.joblib
Shortlist: ['C_ovr/single', 'D_hierarchy/single']


,family,mean_macro_f1,sd_macro_f1,mean_recall_path,mean_ap_path,mean_log_loss,mean_file_macro_f1,shortlisted
0,C_ovr,0.8744,0.0527,0.8623,0.9643,0.2117,0.8950,True
1,D_hierarchy,0.8724,0.0364,0.9001,0.9596,0.2046,0.9022,True
2,B_balanced,0.8701,0.0522,0.8898,0.9450,0.2428,0.9019,False
3,A_multiclass,0.8586,0.0613,0.8435,0.9394,0.2102,0.8810,False
4,B_sqrt,0.8571,0.0641,0.8521,0.9390,0.2288,0.8915,False
5,logistic,0.7217,0.0675,0.5995,0.7553,0.3181,0.7128,False


,candidate,primary,shortlisted,accuracy,macro_f1,balanced_accuracy,recall_path,precision_path,f1_path,recall_normal,recall_suspect,ap_path,roc_auc_path,log_loss,brier,path_as_normal,specificity_path,file_macro_f1,file_recall_path,file_log_loss,file_ap_path
3,C_ovr/single,True,True,0.9224,0.8538,0.8607,0.9143,0.7619,0.8312,0.9608,0.7069,0.9103,0.9859,0.2143,0.1127,0.0286,0.9744,0.8676,0.8576,0.2408,0.9116
5,D_hierarchy/single,False,True,0.9129,0.8454,0.8519,0.9143,0.8000,0.8533,0.9518,0.6897,0.9403,0.9855,0.2193,0.1222,0.0286,0.9795,0.8589,0.8576,0.2483,0.9136


,candidate,macro_f1,recall_path,macro_f1_low,macro_f1_high,recall_low,recall_high,delta_macro_f1_vs_primary,delta_low,delta_high
0,C_ovr/single,0.8538,0.9143,0.7580,0.9154,0.7028,1.0,0.0000,0.0000,0.0000
1,D_hierarchy/single,0.8454,0.9143,0.7466,0.9061,0.7028,1.0,-0.0083,-0.0351,0.0133


### Кластеризация: поддержка исходными файлами

In [9]:
risk=pd.read_csv(ROOT/'research/experiments/ctg_uci/embeddings/test_risk_summary_with_support.csv')
display(risk[(risk.method=='AE')&(risk.algorithm=='GMM')][['cluster','segments','pathological','path_source_files',
    'path_fraction','path_enrichment_low','path_enrichment_high','limited_path_support']].round(4))
stability=pd.read_csv(ROOT/'research/experiments/ctg_uci/embeddings/stability.csv')
display(stability[(stability.method=='AE')&(stability.algorithm=='GMM')].round(4))
print('Высокая доля патологических сегментов и высокий ARI на одном test не подтверждают устойчивые группы риска.')

Высокая доля патологических сегментов и высокий ARI на одном test не подтверждают устойчивые группы риска.


,cluster,segments,pathological,path_source_files,path_fraction,path_enrichment_low,path_enrichment_high,limited_path_support
2,0,45,20,2,0.4444,0.0000,8.5188,True
3,1,380,15,8,0.0395,0.1767,1.0602,False


,key,method,algorithm,mode,repeat,train_segments,train_files,validation_ari_vs_reference,converged
7,AE_51_04,AE,GMM,seed,0,1360,227,0.0745,True
8,AE_51_04,AE,GMM,seed,1,1360,227,-0.0102,True
9,AE_51_04,AE,GMM,file_subsample,0,1128,181,-0.0218,True
10,AE_51_04,AE,GMM,file_subsample,1,1130,181,-0.0326,True
11,AE_51_04,AE,GMM,file_subsample,2,1091,181,0.5999,True
12,AE_51_04,AE,GMM,file_subsample,3,1089,181,0.7143,True
13,AE_51_04,AE,GMM,file_subsample,4,1055,181,0.1310,True


## Assessment · качество и охват

**Готово для воспроизводимого исследовательского сравнения; использовать с оговорками.** Исправления выполнены. Полнота: проверены все основные этапы, а не все возможные датасеты, seed, гиперпараметры или исторические графики. Счётчики ниже относятся к указанной области, не являются процентом научной истинности. Нулевой остаток наблюдённых дефектов не доказывает отсутствие всех ошибок.

| Категория качества ноутбуков | Наблюдённые остаточные дефекты | Оценка |
|---|---|---|
| Полезность и полнота постановки | 0 / 7 ноутбуков | Данные, поиск, калибровка, классификация, кластеры и аудит доступны; независимая внешняя когорта отсутствует |
| Аналитическая ясность | 0 / 4 обновлённых ноутбуков | Основной estimand сохранён, альтернативный определён; история отделена от текущей процедуры |
| Визуальная согласованность | 0 / 5 новых рисунков | Все новые рисунки проверены визуально; исторические повторяющиеся представления проверялись выборочно |
| Интерактивные dashboard-контролы | N/A | Статические исследовательские ноутбуки |

| Аналитическая корректность | Наблюдённые остаточные дефекты | Оценка |
|---|---|---|
| Источник и уверенность | 0 / 3 опорных контракта | Сырые признаки, экспертные метки, исходный файл как proxy; пациентская независимость не установлена |
| Значения и вычисления | 0 / 8 строк моделей | Все строки test-метрик текущих экспортированных моделей пересчитаны независимо в 05 и по ручным формулам в validation receipt |
| Полнота происхождения | 0 / 3 этапа | Подготовка, настройка и экспорт имеют хэши/версии и сохраняемые таблицы |
| Согласованность артефактов | 0 / 4 обновлённых ноутбуков | Исторические результаты сохраняются и явно помечаются, текущий путь модели указан |
| Качество данных | 0 / 4 контракта | Source alignment, grouped boundaries, missing/domain rules, train-only scaler |
| Поддержка выводов | 0 / 3 главных вывода | Классификация экспертного NSP, отсутствие доказанной уникальной лучшей модели, отсутствие устойчиво подтверждённых риск-кластеров |

## Takeaways

1. «Лучшая модель» здесь означает выбранную без нового test процедуру среди проверенных конфигураций. Не заявляются идеальность, глобальный оптимум или готовность к клиническим решениям.
2. Новый основной bundle и возможная альтернатива содержат проверку входа, нужную подготовку, деревья, правильное усреднение, калибровку и порог. Отдельные `.cbm` не заменяют bundle.
3. Известный test нельзя снова сделать слепым. Для следующего утверждения о переносе нужна независимая когорта с идентификаторами пациенток и клинически определённой целью.

Источники методики: [UCI](https://archive.ics.uci.edu/dataset/193/cardiotocography), [nested CV](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html), [calibration](https://scikit-learn.org/stable/modules/calibration.html), [soft voting](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), [UMAP clustering caveats](https://umap-learn.readthedocs.io/en/latest/clustering.html).